# SEKF cached live loop

Checks the frozen Judge prompt hashes, captures Historian/Judge/Synthesizer JSON once, then sweeps thresholds with leave-one-author-out and writes the section 14 paired-bootstrap table plus the markdown vault. The headline metric is leave-one-author-out dev CV until `datasets/test/` exists. The key stays in Colab secrets. Do not commit it.


In [ ]:
import os
from pathlib import Path

root = Path.cwd()
if not (root / "pyproject.toml").is_file():
    !git clone --branch sai-chetan/review-fixes https://github.com/lash106/sekf--memory-governance.git
    %cd sekf--memory-governance
    !pip install -e .


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["SEKF_LLM_API_KEY"] = userdata.get("SEKF_LLM_API_KEY")
except Exception:
    print("No Colab secret. The live cell will skip and the fake capture will run.")
os.environ.setdefault("SEKF_PROVIDER", "openai")
os.environ.setdefault(
    "SEKF_LLM_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/"
)
os.environ.setdefault("SEKF_LLM_MODEL", "gemini-2.0-flash")
os.environ.setdefault("SEKF_POLICY", "v2")


In [ ]:
from sekf.judge.freeze import assert_prompts_frozen

assert_prompts_frozen()
print("prompt hashes match judge/freeze.py")


In [ ]:
import os
from pathlib import Path

out = Path("experiments/runs/colab/predictions.jsonl")
live = os.getenv("SEKF_PROVIDER") == "openai" and bool(os.getenv("SEKF_LLM_API_KEY"))
flag = "--live" if live else ""
print("live" if live else "offline fake capture")
!python scripts/capture_live_predictions.py {flag} --out {out}


In [ ]:
from pathlib import Path

predictions = Path("experiments/runs/colab/predictions.jsonl")
!python scripts/sweep_policy_thresholds.py {predictions}
!python scripts/analyze_judge_evidence.py {predictions}
!python scripts/run_offline_experiment.py --predictions {predictions} --out experiments/offline_cached.json --markdown experiments/vault
print("vault", sorted(p.name for p in Path("experiments/vault").glob("*.md")))
